In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('data/amazon_delivery.csv')

df.head()

df.info()

df.describe(include='all')

df.isnull().sum().sort_values(ascending=False)



<class 'pandas.DataFrame'>
RangeIndex: 43739 entries, 0 to 43738
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Order_ID         43739 non-null  str    
 1   Agent_Age        43739 non-null  int64  
 2   Agent_Rating     43685 non-null  float64
 3   Store_Latitude   43739 non-null  float64
 4   Store_Longitude  43739 non-null  float64
 5   Drop_Latitude    43739 non-null  float64
 6   Drop_Longitude   43739 non-null  float64
 7   Order_Date       43739 non-null  str    
 8   Order_Time       43739 non-null  str    
 9   Pickup_Time      43739 non-null  str    
 10  Weather          43648 non-null  str    
 11  Traffic          43739 non-null  str    
 12  Vehicle          43739 non-null  str    
 13  Area             43739 non-null  str    
 14  Delivery_Time    43739 non-null  int64  
 15  Category         43739 non-null  str    
dtypes: float64(5), int64(2), str(9)
memory usage: 8.6 MB


Weather            91
Agent_Rating       54
Order_ID            0
Agent_Age           0
Store_Latitude      0
Store_Longitude     0
Drop_Latitude       0
Drop_Longitude      0
Order_Date          0
Order_Time          0
Pickup_Time         0
Traffic             0
Vehicle             0
Area                0
Delivery_Time       0
Category            0
dtype: int64

# Basic Cleaning

- Strip leading/trailing spaces
- Store time columns as strings
- Clean missing values

In [11]:
# Strip leading/trailing spaces from all string/object columns
str_cols = df.select_dtypes(include=["object", "string"]).columns
for col in str_cols:
    df[col] = df[col].astype(str).str.strip()

# Stored time columns as strings
time_cols = ["Order_Time", "Pickup_Time"]
for col in time_cols:
    if col in df.columns:
        df[col] = df[col].astype("string")

# Clean missing values
if "Weather" in df.columns:
    df["Weather"] = df["Weather"].fillna("Unknown")

if "Agent_Rating" in df.columns:
    median_rating = df["Agent_Rating"].median()
    df["Agent_Rating"] = df["Agent_Rating"].fillna(median_rating)

# Quick check after cleaning
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 43739 entries, 0 to 43738
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Order_ID         43739 non-null  str    
 1   Agent_Age        43739 non-null  int64  
 2   Agent_Rating     43739 non-null  float64
 3   Store_Latitude   43739 non-null  float64
 4   Store_Longitude  43739 non-null  float64
 5   Drop_Latitude    43739 non-null  float64
 6   Drop_Longitude   43739 non-null  float64
 7   Order_Date       43739 non-null  str    
 8   Order_Time       43739 non-null  string 
 9   Pickup_Time      43739 non-null  string 
 10  Weather          43739 non-null  str    
 11  Traffic          43739 non-null  str    
 12  Vehicle          43739 non-null  str    
 13  Area             43739 non-null  str    
 14  Delivery_Time    43739 non-null  int64  
 15  Category         43739 non-null  str    
dtypes: float64(5), int64(2), str(7), string(2)
memory usage: 8.5 MB


In [ ]:
# Compute pickup delay in minutes
order_ts = pd.to_datetime(df["Order_Date"].astype(str).str.strip() + " " + df["Order_Time"].astype(str).str.strip(), errors="coerce")
pickup_ts = pd.to_datetime(df["Order_Date"].astype(str).str.strip() + " " + df["Pickup_Time"].astype(str).str.strip(), errors="coerce")

df["pickup_delay"] = (pickup_ts - order_ts).dt.total_seconds() / 60

# Utilize Haversine formula to calculate distance between pickup and delivery locations
def calculate_distance(lat1, lon1, lat2, lon2):
    # Convert degrees to radians
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    return 6371 * c  

# Create a new column for distance from store to drop location
df["distance"] = df.apply(
    lambda row: calculate_distance(
        row["Store_Latitude"],
        row["Store_Longitude"],
        row["Drop_Latitude"],
        row["Drop_Longitude"],
    ),
    axis=1,
)

display(df.head())

,Order_ID,Agent_Age,Agent_Rating,Store_Latitude,Store_Longitude,Drop_Latitude,Drop_Longitude,Order_Date,Order_Time,Pickup_Time,Weather,Traffic,Vehicle,Area,Delivery_Time,Category,pickup_delay,distance
0,ialx566343618,37,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,11:30:00,11:45:00,Sunny,High,motorcycle,Urban,120,Clothing,15.0,3.025149
1,akqg208421122,34,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,19:45:00,19:50:00,Stormy,Jam,scooter,Metropolitian,165,Electronics,5.0,20.183530
2,njpu434582536,23,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,08:30:00,08:45:00,Sandstorms,Low,motorcycle,Urban,130,Sports,15.0,1.552758
3,rjto796129700,38,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,18:00:00,18:10:00,Sunny,Medium,motorcycle,Metropolitian,105,Cosmetics,10.0,7.790401
4,zguw716275638,32,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,13:30:00,13:45:00,Cloudy,High,scooter,Metropolitian,150,Toys,15.0,6.210138


# Data Quality Check

In [ ]:
quality = pd.DataFrame({
    "metric": [
        "rows_total",
        "delivery_time_nulls", "delivery_time_nonpositive",
        "distance_nulls", "distance_negative",
        "pickup_delay_nulls", "pickup_delay_negative"
    ],
    "value": [
        len(df),
        df["Delivery_Time"].isna().sum(), (df["Delivery_Time"] <= 0).sum(),
        df["distance"].isna().sum(), (df["distance"] < 0).sum(),
        df["pickup_delay"].isna().sum(), (df["pickup_delay"] < 0).sum()
    ]
})

df_clean = df.copy()
df_clean = df_clean[df_clean["Delivery_Time"].notna() & (df_clean["Delivery_Time"] > 0)]
df_clean = df_clean[df_clean["distance"].notna() & (df_clean["distance"] >= 0)]

display(quality)
print("Rows after cleaning:", len(df_clean))

KeyError: 'distance_km'